In [25]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

df = pd.read_csv("D:\Hoc-May\Lam_Bai_Lab\LAB_1\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.info()
df.head()
df_data = df.drop(['customerID'],axis = 1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [26]:
print(df_data['gender'].unique())
print(df_data['Partner'].unique())
print(df_data['Dependents'].unique())
print(df_data['PhoneService'].unique())
print(df_data['MultipleLines'].unique())
print(df_data['InternetService'].unique())
print(df_data['OnlineSecurity'].unique())
print(df_data['OnlineBackup'].unique())
print(df_data['DeviceProtection'].unique())
print(df_data['TechSupport'].unique())
print(df_data['StreamingTV'].unique())
print(df_data['StreamingMovies'].unique())
print(df_data['Contract'].unique())
print(df_data['PaymentMethod'].unique())
print(df_data['PaperlessBilling'].unique())
print(df_data['TotalCharges'].unique())
print(df_data['Churn'].unique())

['Female' 'Male']
['Yes' 'No']
['No' 'Yes']
['No' 'Yes']
['No phone service' 'No' 'Yes']
['DSL' 'Fiber optic' 'No']
['No' 'Yes' 'No internet service']
['Yes' 'No' 'No internet service']
['No' 'Yes' 'No internet service']
['No' 'Yes' 'No internet service']
['No' 'Yes' 'No internet service']
['No' 'Yes' 'No internet service']
['Month-to-month' 'One year' 'Two year']
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
['Yes' 'No']
['29.85' '1889.5' '108.15' ... '346.45' '306.6' '6844.5']
['No' 'Yes']


In [27]:
df_data['gender'] = df_data['gender'].map({'Male':1, 'Female':0})

binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df_data[col] = df_data[col].map({'Yes':1,'No':0})
    

service_cols = [
    'OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies'
]

for col in service_cols:
    df_data[col] = df_data[col].replace({
        'Yes': 1,
        'No': 0,
        'No internet service': 0
    })
    
df_data['MultipleLines'] = df_data['MultipleLines'].replace({
    'Yes': 1,
    'No': 0,
    'No phone service': 0
})

cat_cols = ['InternetService','Contract','PaymentMethod']

df_data = pd.get_dummies(df_data, columns=cat_cols, drop_first=True)

df_data['TotalCharges'] = pd.to_numeric(df_data['TotalCharges'], errors='coerce')
df_data = df_data.dropna()
df_data.head()



C:\Users\hieum\AppData\Local\Temp\ipykernel_10580\2352912621.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_data[col] = df_data[col].replace({
C:\Users\hieum\AppData\Local\Temp\ipykernel_10580\2352912621.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_data['MultipleLines'] = df_data['MultipleLines'].replace({


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,1,0,1,...,56.95,1889.50,0,False,False,True,False,False,False,True
2,1,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,False,False,False,False,False,False,True
3,1,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,False,False,True,False,False,False,False
4,0,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,True,False,False,False,False,True,False


In [28]:
from sklearn.model_selection import train_test_split

x = df_data.drop('Churn',axis = 1)
y = df_data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

models = {
    "Logistic": LogisticRegression(C=0.1, max_iter=1000,class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=10),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=7,class_weight='balanced'),
    "Dummy": DummyClassifier(strategy="most_frequent")
}

In [30]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)
import numpy as np

def precision_at_k(y_true, y_score, k=200):
    k = min(k, len(y_score))
    top_idx = np.argsort(y_score)[::-1][:k]
    return y_true.iloc[top_idx].sum() / k

# Baseline: predict all non-churn (most frequent class)
baseline_pred = np.zeros_like(y_test)
baseline_prob = np.full(len(y_test), y_train.mean())

baseline_metrics = {
    "Model": "Baseline (churn rate)",
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred, zero_division=0),
    "Recall": recall_score(y_test, baseline_pred),
    "F1": f1_score(y_test, baseline_pred),
    "PR_AUC": average_precision_score(y_test, baseline_prob),
    "Precision@200": precision_at_k(y_test, baseline_prob, 200),
    "ROC_AUC": roc_auc_score(y_test, baseline_prob)
}

results = [baseline_metrics]

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "PR_AUC": average_precision_score(y_test, y_prob) if y_prob is not None else None,
        "Precision@200": precision_at_k(y_test, y_prob, 200) if y_prob is not None else None,
        "ROC_AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else None
    })

df_results = pd.DataFrame(results).sort_values(by="F1", ascending=False)
print(df_results)

c:\Users\hieum\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                   Model  Accuracy  Precision    Recall        F1    PR_AUC  \
3          Random Forest  0.746269   0.514731  0.794118  0.624606  0.647240   
1               Logistic  0.730633   0.495854  0.799465  0.612078  0.624579   
2          Decision Tree  0.778252   0.580729  0.596257  0.588391  0.604350   
0  Baseline (churn rate)  0.734186   0.000000  0.000000  0.000000  0.265814   
4                  Dummy  0.734186   0.000000  0.000000  0.000000  0.265814   

   Precision@200   ROC_AUC  
3          0.665  0.841009  
1          0.690  0.837078  
2          0.660  0.819709  
0          0.235  0.500000  
4          0.235  0.500000  


In [31]:
results_df = pd.DataFrame(results)
print(results_df)

                   Model  Accuracy  Precision    Recall        F1    PR_AUC  \
0  Baseline (churn rate)  0.734186   0.000000  0.000000  0.000000  0.265814   
1               Logistic  0.730633   0.495854  0.799465  0.612078  0.624579   
2          Decision Tree  0.778252   0.580729  0.596257  0.588391  0.604350   
3          Random Forest  0.746269   0.514731  0.794118  0.624606  0.647240   
4                  Dummy  0.734186   0.000000  0.000000  0.000000  0.265814   

   Precision@200   ROC_AUC  
0          0.235  0.500000  
1          0.690  0.837078  
2          0.660  0.819709  
3          0.665  0.841009  
4          0.235  0.500000  


In [32]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import joblib
import os

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
preprocessor = ColumnTransformer(
    [("num", StandardScaler(), numeric_features)],
    remainder="passthrough"
)

best_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced"))
])

best_pipe.fit(X_train, y_train)

os.makedirs("models", exist_ok=True)
joblib.dump(best_pipe, "models/churn_pipeline.joblib")

loaded_pipe = joblib.load("models/churn_pipeline.joblib")
sample_new = X_test.sample(5, random_state=42)
print("Predictions:", loaded_pipe.predict(sample_new))
print("Probabilities:", loaded_pipe.predict_proba(sample_new)[:, 1])

Predictions: [1 0 1 0 0]
Probabilities: [0.87366439 0.09538246 0.53204251 0.07998891 0.00928172]
